# S04.2 – Target leakage: признак, который знает ответ

Цель: показать, что один 'шпионский' признак может сделать метрики почти идеальными – и это не успех, а катастрофа протокола.

## Что вы освоите
- Понять, что такое target leakage и почему это похоже на уязвимость
- Воспроизвести 'улёт метрик' на синтетическом примере
- Научиться базовым приёмам диагностики подозрительных признаков

## Важные оговорки
- Мы моделируем leakage безопасно: синтетически, без атак на реальные системы/данные.
- Смысл – в дисциплине данных и протокола.

---


## Среда, воспроизводимость и стиль эксперимента

Перед кодом – несколько правил, которые будут повторяться во всех ноутбуках:

1) **Воспроизводимость**: фиксируем `random_state` / seed.  
2) **Разделение данных**: test‑часть – это *священная зона*. Мы смотрим на неё только в конце.  
3) **Всё, что "обучается" (`.fit`)** должно видеть только train‑данные (иначе легко получить утечки).  
4) **Sanity‑checks**: после каждого шага проверяем, что получился ожидаемый результат (формы, распределения, пересечения и т.д.).


In [1]:
# Импорты: минимальный, но достаточный набор
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

# Для красивых картинок (простая визуализация)
import matplotlib.pyplot as plt

# Фиксируем seed для воспроизводимости
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("numpy:", np.__version__)
print("pandas:", pd.__version__)


numpy: 2.0.2
pandas: 2.2.2


## Общие функции (оценка моделей и печать метрик)

Чтобы не копировать одно и то же вручную, заведём пару функций.

Важно: эти функции *ничего не делают магически*. Мы специально пишем их максимально прозрачно,
чтобы вы видели, какие именно метрики считаются и на каких данных.


In [2]:
def summarize_binary_metrics(y_true, y_pred, *, positive_label=1):
    """Считает базовые метрики бинарной классификации.

    Мы считаем:
    - accuracy: доля верных ответов
    - precision: насколько "чистые" наши позитивные предсказания
    - recall: насколько хорошо мы находим позитивный класс
    - f1: гармоническое среднее precision и recall

    Почему это важно: в задачах безопасности цена FP и FN может быть разной.
    """
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, pos_label=positive_label, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, pos_label=positive_label, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, pos_label=positive_label, zero_division=0)),
    }

def print_confusion(y_true, y_pred, labels=(0,1)):
    """Печать матрицы ошибок и пояснения, что есть что."""
    cm = confusion_matrix(y_true, y_pred, labels=list(labels))
    df = pd.DataFrame(cm, index=[f"true_{l}" for l in labels], columns=[f"pred_{l}" for l in labels])
    display(df)
    tn, fp, fn, tp = cm.ravel()
    print(f"TN={tn} (верно: 0), FP={fp} (ложная тревога), FN={fn} (пропуск), TP={tp} (верно: 1)")
    return cm

def evaluate_model(model, X_train, y_train, X_test, y_test, *, model_name="model"):
    """Обучает модель на train и оценивает на test. Возвращает словарь метрик."""
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    metrics = summarize_binary_metrics(y_test, y_pred)
    print(f"=== {model_name} ===")
    print(metrics)
    print("Confusion matrix:")
    _ = print_confusion(y_test, y_pred)
    return metrics


## Идея эксперимента

Мы сгенерируем нормальный табличный датасет для бинарной классификации.
Потом добавим **утекающий признак** `leak`, который почти равен целевой метке `y`.

Это аналог ситуации:
- "в признаки попало поле, вычисленное после разметки",
- "в признаки попал идентификатор, кодирующий ответ",
- "в признаки попала статистика, посчитанная по всем данным с использованием y".

Результат: метрики становятся почти идеальными.

Важно: это **не** означает, что модель стала "умной".
Это означает, что протокол и данные скомпрометированы.


In [3]:
from sklearn.datasets import make_classification
from sklearn.tree import DecisionTreeClassifier

# Нормальный датасет
X, y = make_classification(
    n_samples=4000,
    n_features=12,
    n_informative=4,
    n_redundant=2,
    weights=[0.7, 0.3],
    flip_y=0.03,
    random_state=RANDOM_STATE
)
X = pd.DataFrame(X, columns=[f"f{i}" for i in range(X.shape[1])])
y = pd.Series(y, name="y")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

# Базовая простая модель
model = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

base = summarize_binary_metrics(y_test, y_pred)
print("Baseline (no leakage):", base)
print_confusion(y_test, y_pred)


Baseline (no leakage): {'accuracy': 0.911, 'precision': 0.9163498098859315, 'recall': 0.7824675324675324, 'f1': 0.8441330998248686}


,pred_0,pred_1
true_0,670,22
true_1,67,241


TN=670 (верно: 0), FP=22 (ложная тревога), FN=67 (пропуск), TP=241 (верно: 1)


array([[670,  22],
       [ 67, 241]])

## Добавляем утечку: `leak ≈ y`

Мы создаём новый признак:
- `leak = y + маленький шум`
и добавляем его в X.

Обратите внимание: мы делаем это **до** split. Это имитирует ситуацию, когда признак присутствует в исходных данных.


In [4]:
# Создаём leakage-признак
rng = np.random.default_rng(RANDOM_STATE)
leak = y.values + rng.normal(0, 0.02, size=len(y))   # почти y, но не идеально

X_leak = X.copy()
X_leak["leak"] = leak

Xl_train, Xl_test, yl_train, yl_test = train_test_split(
    X_leak, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

model2 = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE)
model2.fit(Xl_train, yl_train)
y_pred2 = model2.predict(Xl_test)

leaky = summarize_binary_metrics(yl_test, y_pred2)
print("With target leakage:", leaky)
print_confusion(yl_test, y_pred2)


With target leakage: {'accuracy': 1.0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0}


,pred_0,pred_1
true_0,692,0
true_1,0,308


TN=692 (верно: 0), FP=0 (ложная тревога), FN=0 (пропуск), TP=308 (верно: 1)


array([[692,   0],
       [  0, 308]])

Ожидаемое наблюдение:
- метрики резко выросли (часто до ~1.0),
- ошибок почти нет.

Это не "мы победили".
Это "мы сломали эксперимент".


## Диагностика: подозрительные признаки

Один из простых приёмов – посмотреть, какие признаки слишком хорошо коррелируют с y
(но делать это нужно **на train**, чтобы не подсматривать test).

Мы посчитаем корреляцию (упрощённый сигнал).


In [5]:
# Корреляция признаков с y (на train!)
corr = Xl_train.assign(y=yl_train).corr(numeric_only=True)["y"].drop("y").abs().sort_values(ascending=False)
display(corr.head(10))
print("Top suspicious feature:", corr.index[0], "corr=", corr.iloc[0])


,y
leak,0.999066
f4,0.315029
f8,0.304338
f11,0.220134
f9,0.064743
f5,0.027570
f10,0.019394
f7,0.014365
f2,0.012273
f6,0.010912


Top suspicious feature: leak corr= 0.9990659298309468


Если вы видите признак с "аномально высокой" связью с y,
это повод задать вопросы:
- откуда этот признак?
- не вычисляется ли он с использованием разметки?
- не содержит ли он будущее/идентификатор/результат?

В реальных проектах диагностика сложнее, но принцип тот же: **подозрительное качество** – это тоже сигнал риска.


## Мини-итог S04.2

1) Target leakage делает метрики "идеальными", но это фикция.  
2) Это похоже на уязвимость: система кажется надёжной, но на самом деле тест скомпрометирован.  
3) Нужно иметь процедуры поиска leakage: анализ происхождения признаков, проверки пересечений, контроль пайплайна.

В конце серии S04 мы соберём чеклист анти‑leakage.
